# 8 - MLP Text Classification: Neural News Classifier

## Learning Objectives

By the end of this notebook, you will be able to:
1. Set up a multi-class text classification problem with PyTorch.
2. Train Word2Vec embeddings on a domain-specific corpus using gensim.
3. Transfer pretrained Word2Vec weights into a PyTorch `nn.Embedding` layer.
4. Design and implement an MLP architecture for text classification.
5. Implement training with validation monitoring and manual early stopping.
6. Evaluate a neural classifier with accuracy, F1, and confusion matrices.
7. Save and reload trained models for inference.

## Prerequisites

- PyTorch fundamentals (tensors, autograd, `nn.Module`, `DataLoader`).
- Text preprocessing basics (tokenization, vocabulary building).
- Basic neural network concepts (forward pass, backpropagation, loss functions).

## Session Format

- **Theory** -> **Demo** -> **Lab** structure.
- Public datasets only (CNN Headlines from Dropbox).
- Target environment: **Google Colab** (GPU recommended but not required).

## The Scenario

You are a data scientist at a news aggregator that receives about 130,000 CNN headlines per week. The product needs to route each headline to one of four sections: Sports (game results, player trades, tournaments), Business (markets, earnings, economic policy), Sci/Tech (research, product launches, space exploration), and World (international politics, conflicts, diplomacy).

An earlier TF-IDF + Logistic Regression baseline reaches about 90% accuracy. The open question is whether a neural network can match or beat that number and hold up on less predictable phrasings.

The goal of this notebook is to build a feed-forward Multi-Layer Perceptron (MLP) that uses Word2Vec embeddings trained on the news corpus itself, reaches test accuracy of at least 90%, and stays reasonable on hand-typed adversarial examples.

The plan: train Word2Vec with gensim, build a PyTorch MLP classifier, train with early stopping, and evaluate against the baseline.

## Section 1: Environment Setup & Hyperparameters

In [ ]:
# Install required packages (run this first in Google Colab)
# If running locally with conda/venv, you may skip this cell

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install gensim textblob scikit-learn pandas matplotlib seaborn

In [ ]:
# Import all necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import random
import os
from collections import Counter
from textblob import TextBlob
import gensim.downloader
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Check PyTorch version
print(f"PyTorch version: {torch.__version__}")

# Device configuration - automatically use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\nEnvironment setup complete!")

In [ ]:
# Hyperparameters - all tunable in one place
SEED = 42
CORPUS_SIZE = 50_000              # Subsample for faster training; full dataset is ~130K
VOCAB_MIN_FREQ = 5                # Words appearing fewer than this are treated as <unk>
MAX_LEN = 30                      # Headlines are short; pad/truncate to this length
EMBEDDING_DIM = 100               # Word2Vec embedding size
HIDDEN_DIM = 128                  # Hidden layer dimension in MLP
N_CLASSES = 4                     # Sports, Business, Sci/Tech, World
DROPOUT = 0.3                     # Dropout probability for regularization
BATCH_SIZE = 128                  # Training batch size
EPOCHS_MAX = 15                   # Maximum training epochs
LR = 1e-3                         # Learning rate for Adam optimizer
EARLY_STOP_PATIENCE = 3           # Stop if val accuracy doesn't improve for this many epochs

# Set random seeds for reproducibility
def set_seed(seed=42):
    """Set random seeds for reproducibility across PyTorch, NumPy, and Python."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Hyperparameters configured and seeds set for reproducibility.")

In [ ]:
# Load CNN Headlines dataset
import urllib.request

# Download the dataset
url = "https://www.dropbox.com/s/352x7xzivf60zgc/news.csv?dl=1"
print("Downloading CNN Headlines dataset...")
urllib.request.urlretrieve(url, "news.csv")
print("Download complete!")

# Load into pandas DataFrame
df = pd.read_csv("news.csv")

# Subsample for faster training (you can use full dataset by setting CORPUS_SIZE = len(df))
if CORPUS_SIZE < len(df):
    df = df.sample(n=CORPUS_SIZE, random_state=SEED).reset_index(drop=True)
    print(f"Sampled {CORPUS_SIZE} headlines from {len(pd.read_csv('news.csv'))} total.")
else:
    print(f"Using full dataset: {len(df)} headlines.")

# Inspect the dataset
print("\n=== Dataset Overview ===")
print(df.head(10))
print(f"\nShape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\n=== Class Distribution ===")
print(df['category'].value_counts())
print(f"\n=== Sample Headlines by Category ===")
for category in df['category'].unique():
    sample = df[df['category'] == category].iloc[0]['title']
    print(f"{category:12} | {sample}")

## Section 2: Text Preprocessing & Word2Vec Embeddings

### The Baseline Bar

The TF-IDF + Logistic Regression baseline reaches **~90% accuracy** on this 4-class problem. That is a solid number, but it treats each word as an independent feature. Words like "stock", "stocks", and "stockholder" each get their own TF-IDF dimension with no shared representation.

**Why Word2Vec might help.** Word2Vec learns that semantically similar words (for example, "stock", "shares", "equity") should have similar vector representations. That usually helps with:
1. **Generalization** across unseen word combinations in test data.
2. **Robustness** to typos or rare word forms.
3. **Semantic similarity** so that "market crashes" and "stocks plummet" share meaning.

### The Plan

1. **Tokenize** all headlines with TextBlob (same as in previous notebooks).
2. **Train Word2Vec** on the tokenized corpus using gensim.
3. **Build a vocabulary** mapping words to integer IDs.
4. **Create an embedding matrix** to initialize our `nn.Embedding` layer.

In [ ]:
# Tokenize headlines with TextBlob
import re

def preprocess_text(text):
    """
    Clean and tokenize text.
    
    Steps:
    1. Convert to lowercase
    2. Remove special characters (keep only letters and spaces)
    3. Tokenize with TextBlob
    4. Return list of tokens
    """
    # Lowercase
    text = text.lower()
    # Remove special characters and digits
    text = re.sub(r"[^a-z\s]", " ", text)
    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Tokenize with TextBlob
    tokens = [word for word in TextBlob(text).words]
    return tokens

# Apply to all headlines
print("Tokenizing headlines...")
df['tokens'] = df['title'].apply(preprocess_text)

# Show examples
print("\n=== Tokenization Examples ===")
for i in range(3):
    print(f"Original : {df.iloc[i]['title']}")
    print(f"Tokenized: {df.iloc[i]['tokens']}")
    print()

# Build corpus as list of token lists for Word2Vec
corpus = df['tokens'].tolist()
print(f"Corpus ready: {len(corpus)} documents")

In [ ]:
# Train Word2Vec on the news corpus
print("Training Word2Vec model...")
print(f"Parameters: vector_size={EMBEDDING_DIM}, window=5, min_count={VOCAB_MIN_FREQ}, workers=2, epochs=10")

# Train Word2Vec with gensim
# - vector_size: dimensionality of word vectors (matches EMBEDDING_DIM)
# - window: how many words around the target word to consider as context
# - min_count: ignore words appearing fewer than this (helps reduce vocab size)
# - workers: number of CPU threads (set to 2 for Colab stability)
# - epochs: how many passes over the corpus
w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=VOCAB_MIN_FREQ,
    workers=2,
    epochs=10,
    seed=SEED
)

print(f"Word2Vec training complete!")
print(f"Vocabulary size: {len(w2v_model.wv)} words")
print(f"Embedding shape: ({len(w2v_model.wv)}, {EMBEDDING_DIM})")

In [ ]:
# Sanity check: are the embeddings sensible?
print("=== Word2Vec Sanity Check ===")
print("Testing semantic similarity...\n")

# Test 1: Financial words
test_words = ['stock', 'game', 'software', 'war']
for word in test_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"'{word}' is similar to: {[w for w, score in similar]}")
    else:
        print(f"'{word}' not in vocabulary")

# Test 2: Check a specific embedding vector
if 'market' in w2v_model.wv:
    vec = w2v_model.wv['market']
    print(f"\n'market' embedding vector (first 10 dims): {vec[:10]}")
    print(f"Vector shape: {vec.shape}")
else:
    print("\n'market' not in vocabulary")

In [ ]:
# Build vocabulary: word -> integer ID mapping
# Special tokens:
#   <pad> (ID=0): for padding shorter sequences to MAX_LEN
#   <unk> (ID=1): for words not in Word2Vec vocabulary

def build_vocab(w2v_model):
    """
    Build word-to-index and index-to-word mappings from Word2Vec model.
    Reserve index 0 for <pad> and index 1 for <unk>.
    """
    word_to_idx = {'<pad>': 0, '<unk>': 1}
    idx_to_word = {0: '<pad>', 1: '<unk>'}
    
    # Add Word2Vec vocabulary starting from index 2
    for idx, word in enumerate(w2v_model.wv.index_to_key, start=2):
        word_to_idx[word] = idx
        idx_to_word[idx] = word
    
    return word_to_idx, idx_to_word

word_to_idx, idx_to_word = build_vocab(w2v_model)
vocab_size = len(word_to_idx)

print(f"Vocabulary built: {vocab_size} tokens")
print(f"Special tokens: {list(word_to_idx.keys())[:2]}")
print(f"First 10 words: {list(word_to_idx.keys())[2:12]}")

In [ ]:
# Build embedding matrix from Word2Vec
# Shape: (vocab_size, EMBEDDING_DIM)
# Row i contains the embedding vector for word with ID i

def build_embedding_matrix(w2v_model, word_to_idx, embedding_dim):
    """
    Create embedding matrix from Word2Vec model.
    
    - Row 0 (<pad>): all zeros (enforced later)
    - Row 1 (<unk>): random vector or mean of all embeddings
    - Rows 2+: Word2Vec vectors
    """
    vocab_size = len(word_to_idx)
    embedding_matrix = np.zeros((vocab_size, embedding_dim), dtype=np.float32)
    
    # Initialize <unk> as the mean of all Word2Vec vectors
    embedding_matrix[1] = w2v_model.wv.vectors.mean(axis=0)
    
    # Fill in Word2Vec vectors for words in vocabulary
    for word, idx in word_to_idx.items():
        if word in w2v_model.wv:
            embedding_matrix[idx] = w2v_model.wv[word]
    
    return embedding_matrix

embedding_matrix = build_embedding_matrix(w2v_model, word_to_idx, EMBEDDING_DIM)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"<pad> embedding (should be zeros): {embedding_matrix[0][:10]}")
print(f"<unk> embedding (mean of all): {embedding_matrix[1][:10]}")
print(f"Sample word embedding: {embedding_matrix[100][:10]}")

In [ ]:
# Encode headlines as sequences of integer IDs and create labels
def text_to_ids(tokens, word_to_idx):
    """Convert list of tokens to list of integer IDs."""
    return [word_to_idx.get(token, 1) for token in tokens]  # 1 = <unk>

def pad_or_truncate(ids, max_len, pad_idx=0):
    """Pad short sequences or truncate long ones to max_len."""
    if len(ids) < max_len:
        ids = ids + [pad_idx] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

# Convert tokens to IDs
df['ids'] = df['tokens'].apply(lambda tokens: text_to_ids(tokens, word_to_idx))

# Pad/truncate to MAX_LEN
df['ids_padded'] = df['ids'].apply(lambda ids: pad_or_truncate(ids, MAX_LEN))

# Create label mapping: category name -> integer
category_to_label = {
    'Business': 0,
    'Sports': 1,
    'Sci/Tech': 2,
    'World': 3
}
df['label'] = df['category'].map(category_to_label)

# Show examples
print("=== Encoding Examples ===")
for i in range(2):
    print(f"Tokens     : {df.iloc[i]['tokens']}")
    print(f"IDs (raw)  : {df.iloc[i]['ids']}")
    print(f"IDs (padded): {df.iloc[i]['ids_padded']}")
    print(f"Label      : {df.iloc[i]['label']} ({df.iloc[i]['category']})")
    print()

### Lab 1: Build a PyTorch Dataset

**Task:** Implement a custom `HeadlineDataset` class that:
1. Takes the preprocessed DataFrame as input
2. Returns `(input_ids, mask, label)` for each sample in `__getitem__`
   - `input_ids`: padded sequence of token IDs (shape: `[MAX_LEN]`)
   - `mask`: binary mask (1 for real tokens, 0 for padding) (shape: `[MAX_LEN]`)
   - `label`: integer class label (0-3)
3. Implements `__len__` to return the dataset size

**Why the mask?** When we compute the mean of word embeddings, we want to average only the real words, not the padding tokens. The mask tells us which positions are real.

**Starter code provided below. Fill in the placeholders marked with `None # YOUR CODE`.**

In [ ]:
# Lab 1: Implement HeadlineDataset
class HeadlineDataset(Dataset):
    """PyTorch Dataset for news headlines."""
    
    def __init__(self, dataframe):
        """
        Args:
            dataframe: pandas DataFrame with columns 'ids_padded' and 'label'
        """
        self.ids = dataframe['ids_padded'].tolist()
        self.labels = dataframe['label'].tolist()
    
    def __len__(self):
        """Return the number of samples in the dataset."""
        return None  # YOUR CODE: return dataset size
    
    def __getitem__(self, idx):
        """
        Return a single sample.
        
        Returns:
            input_ids: torch.LongTensor of shape [MAX_LEN]
            mask: torch.FloatTensor of shape [MAX_LEN] (1 for real tokens, 0 for pad)
            label: torch.LongTensor (scalar)
        """
        # Get the token IDs for this sample
        input_ids = None  # YOUR CODE: get token IDs from self.ids[idx]
        
        # Create mask: 1 where token != 0 (<pad>), 0 where token == 0
        mask = None  # YOUR CODE: create binary mask as a list comprehension
        
        # Get label
        label = None  # YOUR CODE: get label from self.labels[idx]
        
        # Convert to PyTorch tensors
        input_ids = torch.LongTensor(input_ids)
        mask = torch.FloatTensor(mask)
        label = torch.LongTensor([label])
        
        return input_ids, mask, label

# Test your implementation
test_dataset = HeadlineDataset(df.head(5))
print(f"Dataset size: {len(test_dataset)}")
test_ids, test_mask, test_label = test_dataset[0]
print(f"Sample 0:")
print(f"  Input IDs shape: {test_ids.shape}")
print(f"  Mask shape: {test_mask.shape}")
print(f"  Label: {test_label.item()}")

## Section 3: MLP Architecture for Text Classification

### The Architecture

Our MLP follows this flow:

```
Input: [B, MAX_LEN] token IDs
  ↓
Embedding: [B, MAX_LEN, EMBEDDING_DIM]
  ↓
Masked Mean Pooling: [B, EMBEDDING_DIM]  ← average only real tokens (not padding)
  ↓
Linear(EMBEDDING_DIM, HIDDEN_DIM): [B, HIDDEN_DIM]
  ↓
ReLU
  ↓
Dropout(0.3)
  ↓
Linear(HIDDEN_DIM, N_CLASSES): [B, 4]  ← logits (raw scores, no softmax)
```

**Key design choices:**
1. **Masked mean pooling**: We average word embeddings but ignore padding (using the mask)
2. **No softmax in forward()**: `nn.CrossEntropyLoss` applies log-softmax internally, so we return raw logits
3. **Pretrained embeddings**: We'll initialize the embedding layer with our Word2Vec matrix
4. **Padding index**: we force row 0 (the `<pad>` token) to stay all zeros during training.

In [ ]:
# Demo: Define the NewsClassifier model
class NewsClassifier(nn.Module):
    """
    Multi-Layer Perceptron for news headline classification.
    
    Architecture:
        Embedding -> Masked Mean Pool -> Linear -> ReLU -> Dropout -> Linear
    """
    
    def __init__(self, embedding_matrix, hidden_dim=128, n_classes=4, 
                 dropout=0.3, freeze_embeddings=False, pad_idx=0):
        """
        Args:
            embedding_matrix: numpy array of shape (vocab_size, embedding_dim)
            hidden_dim: size of hidden layer
            n_classes: number of output classes
            dropout: dropout probability
            freeze_embeddings: if True, don't update embeddings during training
            pad_idx: index of padding token (will be forced to zero vector)
        """
        super().__init__()
        
        vocab_size, embedding_dim = embedding_matrix.shape
        
        # Embedding layer initialized with Word2Vec weights
        self.emb = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix),
            freeze=freeze_embeddings,
            padding_idx=pad_idx
        )
        
        # Feedforward layers
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, n_classes)
    
    def forward(self, input_ids, mask):
        """
        Forward pass.
        
        Args:
            input_ids: [batch_size, max_len] - token IDs
            mask: [batch_size, max_len] - binary mask (1=real token, 0=padding)
        
        Returns:
            logits: [batch_size, n_classes] - raw scores (no softmax)
        """
        # Embed tokens: [B, L] -> [B, L, D]
        embeddings = self.emb(input_ids)
        
        # Masked mean pooling: [B, L, D] -> [B, D]
        # Multiply embeddings by mask to zero out padding, then average
        mask = mask.unsqueeze(-1)  # [B, L, 1] for broadcasting
        masked_embeddings = embeddings * mask  # [B, L, D]
        sum_embeddings = masked_embeddings.sum(dim=1)  # [B, D]
        num_real_tokens = mask.sum(dim=1).clamp(min=1)  # [B, 1], avoid division by zero
        pooled = sum_embeddings / num_real_tokens  # [B, D]
        
        # Feedforward layers
        hidden = torch.relu(self.fc1(pooled))  # [B, hidden_dim]
        hidden = self.dropout(hidden)
        logits = self.fc2(hidden)  # [B, n_classes]
        
        return logits

print("NewsClassifier model defined successfully!")

In [ ]:
# Demo: Count parameters
def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Instantiate model for parameter counting (temporary, we'll rebuild it properly in the lab)
temp_model = NewsClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=HIDDEN_DIM,
    n_classes=N_CLASSES,
    dropout=DROPOUT,
    freeze_embeddings=False
)

total_params, trainable_params = count_parameters(temp_model)
embedding_params = vocab_size * EMBEDDING_DIM
other_params = total_params - embedding_params

print("=== Parameter Count ===")
print(f"Total parameters: {total_params:,}")
print(f"  Embedding layer: {embedding_params:,}")
print(f"  Other layers: {other_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nNote: Most parameters are in the embedding layer!")
print(f"      Freezing embeddings would reduce trainable params to {other_params:,}.")

In [ ]:
# Demo: Test forward pass with dummy data
dummy_ids = torch.randint(0, vocab_size, (4, MAX_LEN))  # batch of 4 headlines
dummy_mask = torch.ones(4, MAX_LEN)  # all real tokens (no padding)
dummy_mask[:, 20:] = 0  # pretend last 10 positions are padding

temp_model.eval()  # set to eval mode (disables dropout)
with torch.no_grad():
    dummy_logits = temp_model(dummy_ids, dummy_mask)

print("=== Dummy Forward Pass ===")
print(f"Input shape: {dummy_ids.shape}")
print(f"Mask shape: {dummy_mask.shape}")
print(f"Output logits shape: {dummy_logits.shape}")
print(f"Sample logits: {dummy_logits[0]}")
print(f"\nNote: These are raw scores (logits), not probabilities.")
print(f"      CrossEntropyLoss will convert them to log-probabilities internally.")

### Lab 2: Instantiate Model and Load Pretrained Weights

**Task:** 
1. Create the `NewsClassifier` model with the correct hyperparameters
2. Move it to the appropriate device (GPU or CPU)
3. Ensure the `<pad>` token (index 0) embedding is forced to all zeros

**Why force padding to zero?** Even though we use a mask during forward pass, forcing the padding embedding to zero provides an extra safeguard. If the mask accidentally lets padding through, the zero vector won't contribute to the mean.

**Fill in the placeholders below.**

In [ ]:
# Lab 2: Instantiate and initialize the model
# Create the model
model = None  # YOUR CODE: instantiate NewsClassifier with embedding_matrix and hyperparameters

# Move to device (GPU/CPU)
# YOUR CODE: move model to device using .to()

# Force padding embedding (index 0) to zero vector
# YOUR CODE: set model.emb.weight.data[0] to a zero vector

print("Model instantiated and initialized!")
print(f"Device: {next(model.parameters()).device}")
print(f"Padding embedding (should be all zeros): {model.emb.weight.data[0][:10]}")

## Section 4: Training Loop with Validation & Early Stopping

### The Training Strategy

We'll split our data into three sets:
- **Train (80%)**: Update model weights
- **Val (10%)**: Monitor performance and decide when to stop
- **Test (10%)**: Final evaluation (touch only once at the end)

**Early stopping:** We track validation accuracy after each epoch. If it doesn't improve for `EARLY_STOP_PATIENCE` consecutive epochs, we stop training and reload the best checkpoint. This prevents overfitting.

**Why stratified split?** Our classes are reasonably balanced but not perfectly equal. Stratified splitting ensures each split has the same class proportions as the full dataset.

In [ ]:
# Train/val/test split (80/10/10, stratified by class)
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label']
)

print("=== Data Split ===")
print(f"Train: {len(train_df)} samples")
print(f"Val:   {len(val_df)} samples")
print(f"Test:  {len(test_df)} samples")

# Create PyTorch Datasets
train_dataset = HeadlineDataset(train_df)
val_dataset = HeadlineDataset(val_df)
test_dataset = HeadlineDataset(test_df)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Define loss function and optimizer
# CrossEntropyLoss: takes logits and integer labels (NOT one-hot)
loss_fn = nn.CrossEntropyLoss()

# Adam optimizer: adaptive learning rate, works well out-of-the-box
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print("=== Training Setup ===")
print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr={LR})")
print(f"Max epochs: {EPOCHS_MAX}")
print(f"Early stop patience: {EARLY_STOP_PATIENCE}")

In [ ]:
# Helper functions for training and evaluation
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """Train for one epoch and return average loss."""
    model.train()  # enable dropout
    total_loss = 0
    
    for input_ids, mask, labels in loader:
        # Move to device
        input_ids = input_ids.to(device)
        mask = mask.to(device)
        labels = labels.to(device).squeeze()  # [B]
        
        # Forward pass
        logits = model(input_ids, mask)
        loss = loss_fn(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, device):
    """Evaluate and return accuracy and loss."""
    model.eval()  # disable dropout
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for input_ids, mask, labels in loader:
            # Move to device
            input_ids = input_ids.to(device)
            mask = mask.to(device)
            labels = labels.to(device).squeeze()
            
            # Forward pass
            logits = model(input_ids, mask)
            loss = loss_fn(logits, labels)
            
            # Predictions
            preds = torch.argmax(logits, dim=-1)
            
            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy

print("Helper functions defined!")

### Lab 3: Implement Training Loop with Early Stopping

**Task:** Complete the training loop below. It should:
1. Train for up to `EPOCHS_MAX` epochs
2. After each epoch, evaluate on the validation set
3. If validation accuracy improves, save the model and reset patience counter
4. If validation accuracy doesn't improve for `EARLY_STOP_PATIENCE` consecutive epochs, stop training
5. After training completes (or early stops), reload the best checkpoint

**Fill in the placeholders marked with `None # YOUR CODE`.**

In [ ]:
# Lab 3: Training loop with early stopping
# Track metrics
history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': []
}

# Early stopping variables
best_val_acc = 0.0
patience_counter = 0
best_model_path = 'best_model.pt'

print("Starting training...\n")

for epoch in range(EPOCHS_MAX):
    # Train for one epoch
    train_loss = None  # YOUR CODE: call train_one_epoch()
    
    # Evaluate on validation set
    val_loss, val_acc = None, None  # YOUR CODE: call evaluate() on val_loader
    
    # Log metrics
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS_MAX} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}")
    
    # Early stopping logic
    if val_acc > best_val_acc:
        # YOUR CODE: update best_val_acc
        # YOUR CODE: reset patience_counter to 0
        # YOUR CODE: save model state_dict to best_model_path
        print(f"  -> Val accuracy improved. Saving checkpoint.")
    else:
        # YOUR CODE: increment patience_counter by 1
        print(f"  -> No improvement (patience: {patience_counter}/{EARLY_STOP_PATIENCE})")
        
        # YOUR CODE: if patience_counter >= EARLY_STOP_PATIENCE, break the loop

print(f"\nTraining finished!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curve (val only)
ax2.plot(history['val_acc'], label='Val Accuracy', marker='s', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Training Summary ===")
print(f"Total epochs run: {len(history['train_loss'])}")
print(f"Best val accuracy: {max(history['val_acc']):.4f} at epoch {np.argmax(history['val_acc'])+1}")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

In [ ]:
# Reload best checkpoint before test evaluation
model.load_state_dict(torch.load(best_model_path))
print("Best model checkpoint reloaded!")
print(f"This model achieved {best_val_acc:.4f} validation accuracy.")
print("\nNow we're ready to evaluate on the test set.")

### Why Evaluate on Test Only Once?

**Important principle:** The test set is your *final exam*. You touch it only once, after all hyperparameter tuning is complete.

**The workflow:**
1. **Train set**: Update model weights
2. **Val set**: Monitor performance, decide when to stop, compare architectures, tune hyperparameters
3. **Test set**: Final evaluation to report to stakeholders

If you repeatedly evaluate on the test set and adjust based on test performance, you're indirectly *training* on the test set. This inflates reported metrics and misleads stakeholders about real-world performance.

In practice: if you ran five experiments with different hyperparameters and picked the best one, report val accuracy for all five plus test accuracy for the winner only.

## Section 5: Evaluation on Test Set

Now that the model is trained and the best checkpoint reloaded, we evaluate on the held-out test set.

We compute:
1. **Accuracy** for overall correctness.
2. **Macro F1**, the average F1 across classes, which handles class imbalance.
3. **Per-class F1** to see which categories are easiest and hardest.
4. **Confusion matrix** to see which classes get confused with each other.

In [ ]:
# Compute test set metrics
test_loss, test_acc = evaluate(model, test_loader, device)

# Get predictions and labels for detailed analysis
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for input_ids, mask, labels in test_loader:
        input_ids = input_ids.to(device)
        mask = mask.to(device)
        labels = labels.to(device).squeeze()
        
        logits = model(input_ids, mask)
        preds = torch.argmax(logits, dim=-1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute metrics
macro_f1 = f1_score(all_labels, all_preds, average='macro')

print("=== Test Set Performance ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print("\n=== Per-Class Report ===")
target_names = ['Business', 'Sports', 'Sci/Tech', 'World']
print(classification_report(all_labels, all_preds, target_names=target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

print("\n=== Confusion Matrix Insights ===")
print("Look for:")
print("  - High numbers on the diagonal (correct predictions)")
print("  - Off-diagonal patterns (which classes get confused?)")
print(f"\nMost confused pair:")
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
max_confusion = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
print(f"  {target_names[max_confusion[0]]} -> {target_names[max_confusion[1]]}: {cm_no_diag[max_confusion]} errors")

In [ ]:
# Inference on new headlines (hand-typed)
def predict_headline(text, model, word_to_idx, device):
    """Predict class for a single headline."""
    # Tokenize and encode
    tokens = preprocess_text(text)
    ids = text_to_ids(tokens, word_to_idx)
    ids_padded = pad_or_truncate(ids, MAX_LEN)
    
    # Create mask
    mask = [1 if token_id != 0 else 0 for token_id in ids_padded]
    
    # Convert to tensors
    input_ids = torch.LongTensor([ids_padded]).to(device)
    mask_tensor = torch.FloatTensor([mask]).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(input_ids, mask_tensor)
        probs = torch.softmax(logits, dim=-1)
        pred_class = torch.argmax(logits, dim=-1).item()
        confidence = probs[0][pred_class].item()
    
    return pred_class, confidence, probs[0].cpu().numpy()

# Test on 5 new headlines
test_headlines = [
    "Stocks plummet after Fed raises interest rates",
    "Scientists discover new exoplanet in habitable zone",
    "Lakers defeat Celtics in overtime thriller",
    "UN condemns military action in conflict zone",
    "Apple announces new iPhone with AI features"
]

label_to_category = {v: k for k, v in category_to_label.items()}

print("=== Inference on New Headlines ===\n")
for headline in test_headlines:
    pred_class, confidence, probs = predict_headline(headline, model, word_to_idx, device)
    predicted_category = label_to_category[pred_class]
    
    print(f"Headline: {headline}")
    print(f"Predicted: {predicted_category} (confidence: {confidence:.3f})")
    print(f"All probabilities: {dict(zip(target_names, probs))}")
    print()

In [ ]:
# Misclassification analysis: find examples where model failed
test_df_copy = test_df.copy().reset_index(drop=True)
test_df_copy['predicted'] = all_preds
test_df_copy['correct'] = test_df_copy['label'] == test_df_copy['predicted']

# Sample 5 errors per class (if available)
print("=== Misclassification Examples ===\n")
for true_label in range(N_CLASSES):
    errors = test_df_copy[(test_df_copy['label'] == true_label) & (~test_df_copy['correct'])]
    
    print(f"True class: {target_names[true_label]} ({len(errors)} errors)")
    
    if len(errors) > 0:
        sample_errors = errors.head(3)
        for _, row in sample_errors.iterrows():
            predicted_class = target_names[row['predicted']]
            print(f"  '{row['title']}'")
            print(f"    -> Predicted as: {predicted_class}")
    print()

### Lab 4: Test on Adversarial Examples

**Task:** Create 3 deliberately ambiguous headlines that could belong to multiple categories. Test your model on them and discuss:
1. What did the model predict?
2. Why might these be hard to classify?
3. Is there a "correct" answer, or are they genuinely ambiguous?

**Examples of ambiguous headlines:**
- "Tech company announces record earnings" (Business or Sci/Tech?)
- "Olympic committee investigates doping scandal" (Sports or World?)
- "Scientists warn of economic impact from climate change" (Sci/Tech, Business, or World?)

**Your turn: Create 3 adversarial headlines and test them below.**

In [ ]:
# Lab 4: Test your adversarial headlines here
adversarial_headlines = [
    None,  # YOUR CODE: add your first adversarial headline
    None,  # YOUR CODE: add your second adversarial headline
    None   # YOUR CODE: add your third adversarial headline
]

# Uncomment and run after filling in headlines
# for headline in adversarial_headlines:
#     pred_class, confidence, probs = predict_headline(headline, model, word_to_idx, device)
#     predicted_category = label_to_category[pred_class]
#     print(f"Headline: {headline}")
#     print(f"Predicted: {predicted_category} (confidence: {confidence:.3f})")
#     print(f"All probabilities: {dict(zip(target_names, probs))}")
#     print()

In [ ]:
# Compare with baseline
print("=== Comparison with Baseline ===")
print(f"TF-IDF + LogReg baseline: ~90% accuracy (from previous work)")
print(f"Our MLP:                  {test_acc:.4f} accuracy")
print(f"Macro F1:                 {macro_f1:.4f}")
print()

if test_acc >= 0.90:
    print("We matched or beat the baseline.")
    print("  The MLP learned to classify news headlines at the target accuracy.")
else:
    print("We are slightly below the baseline.")
    print("  Potential improvements: tune hyperparameters, add more data, or try a different architecture.")

print("\n=== What Did We Gain? ===")
print("Even if accuracy is similar to LogReg, the MLP offers:")
print("1. Semantic understanding via Word2Vec (similar words share representations)")
print("2. Nonlinear decision boundaries (ReLU + hidden layers)")
print("3. Foundation for deeper architectures (CNNs, RNNs, Transformers)")
print("4. Easier to incorporate pretrained embeddings (GloVe, fastText)")

## Section 6: Ablations & Hyperparameter Insights

An **ablation study** answers the question: what happens if we remove or change component X?

We walk through a few architectural choices and their typical impact on performance. These are demonstrations meant to build intuition; actual tuning happens in the lab.

### Key Questions
1. **Freeze embeddings?** Should we update Word2Vec weights during training, or keep them fixed?
2. **Pooling strategy?** Mean vs. max vs. sum?
3. **Hidden layer size?** Does a bigger hidden layer help?
4. **Dropout?** How much regularization do we need?

In [ ]:
# Demo: Effect of freezing embeddings
print("=== Ablation: Freeze vs. Fine-tune Embeddings ===\n")

# This is a discussion of expected trade-offs; in practice you would retrain both
# models fully and compare.

print("Frozen embeddings (freeze=True):")
print("  Training is faster and uses fewer trainable parameters, which reduces")
print("  overfitting risk. The cost is that Word2Vec vectors cannot adapt to the")
print("  specific task, so they have to be a good fit out of the box. Reasonable")
print("  default when the dataset is small and the pretrained vectors are strong.")
print()

print("Fine-tuned embeddings (freeze=False, our choice):")
print("  The embeddings can specialize (for example, 'stock' moves closer to")
print("  business-related words in this corpus). More trainable parameters mean")
print("  higher overfitting risk on small datasets, so this works best with a")
print("  medium-to-large dataset and enough compute.")
print()

print("Typical impact: fine-tuning often improves test accuracy by 1-3 percentage points.")
print(f"Our model: freeze=False -> {test_acc:.4f} accuracy")
print()

# Demo: Different pooling strategies
print("=== Ablation: Pooling Strategies ===\n")
print("Mean pooling (our choice):")
print("  Averages all word embeddings into a smooth, stable representation.")
print("  Fits well when every word carries roughly similar signal.")
print()

print("Max pooling:")
print("  Takes the max value per dimension and highlights salient features.")
print("  Works well when a few keywords (like 'touchdown', 'earnings') dominate.")
print()

print("Sum pooling:")
print("  Sums embeddings, which makes the representation sensitive to sequence")
print("  length. Rarely used unless length is controlled.")
print()

print("For headlines (short, every word matters), mean pooling is a solid choice.")
print()

# Demo: Hidden layer size
print("=== Ablation: Hidden Layer Size ===\n")
print(f"Our choice: HIDDEN_DIM={HIDDEN_DIM}")
print("  Typical range for text MLPs: 64-512.")
print("  Larger -> more capacity but higher overfitting risk.")
print("  Smaller -> faster and more regularized but may underfit.")
print()
print("A common starting point is embedding_dim (100) or 2x (200); tune from there.")
print()

# Demo: Dropout
print("=== Ablation: Dropout Rate ===\n")
print(f"Our choice: DROPOUT={DROPOUT}")
print("  Dropout randomly zeros activations during training, which prevents")
print("  co-adaptation of units.")
print("  Typical range: 0.1-0.5.")
print("  Higher -> more regularization (lower train acc, maybe higher val acc).")
print("  Lower  -> less regularization (higher train acc, risk of overfitting).")
print()
print("For this dataset, 0.3 is a safe default. If train >> val accuracy, push it to 0.4-0.5.")

### Lab 5: Mini Hyperparameter Experiment

**Task:** Pick ONE hyperparameter to vary and measure its effect on validation accuracy. Suggestions:
- `HIDDEN_DIM`: try [64, 128, 256]
- `DROPOUT`: try [0.2, 0.3, 0.5]
- `LR`: try [1e-4, 1e-3, 1e-2]
- `freeze_embeddings`: try [True, False]

**Steps:**
1. Pick a hyperparameter
2. Train 3 models with different values (you can reduce `CORPUS_SIZE` to 10000 for speed)
3. Compare validation accuracy
4. Report your findings

**Note:** This is time-consuming. If you're short on time, just change the hyperparameter value at the top, re-run the training cells, and observe the difference.

## Section 7: Save Model & Wrap-up

To reuse this model later, we need to save:
1. **Model weights** (`state_dict`).
2. **Vocabulary** (the `word_to_idx` mapping).
3. **Hyperparameters** (for reproducibility).

With those artifacts, the model can be reloaded in a new notebook or service without retraining.

In [ ]:
# Save model, vocabulary, and hyperparameters
import pickle

# Save model state dict
torch.save(model.state_dict(), 'news_classifier.pt')

# Save vocabulary
with open('vocab.pkl', 'wb') as f:
    pickle.dump({
        'word_to_idx': word_to_idx,
        'idx_to_word': idx_to_word,
        'category_to_label': category_to_label,
        'label_to_category': label_to_category
    }, f)

# Save hyperparameters
hparams = {
    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'n_classes': N_CLASSES,
    'dropout': DROPOUT,
    'max_len': MAX_LEN
}

with open('hparams.pkl', 'wb') as f:
    pickle.dump(hparams, f)

print("Model saved to 'news_classifier.pt'")
print("Vocabulary saved to 'vocab.pkl'")
print("Hyperparameters saved to 'hparams.pkl'")
print("\n=== How to Reload ===")
print("""
# Load hyperparameters
with open('hparams.pkl', 'rb') as f:
    hparams = pickle.load(f)

# Load vocabulary
with open('vocab.pkl', 'rb') as f:
    vocab_data = pickle.load(f)
    word_to_idx = vocab_data['word_to_idx']
    label_to_category = vocab_data['label_to_category']

# Rebuild embedding matrix (or save it separately)
# embedding_matrix = ...

# Instantiate model
model = NewsClassifier(embedding_matrix, **hparams)
model.load_state_dict(torch.load('news_classifier.pt'))
model.eval()

# Now you can call predict_headline(text, model, word_to_idx, device)
""")

## Wrap-up

### What You Learned

1. Trained Word2Vec embeddings on a domain-specific corpus.
2. Designed an MLP architecture with masked mean pooling.
3. Implemented manual early stopping with validation monitoring.
4. Evaluated with accuracy, F1, and confusion matrices.
5. Analyzed misclassifications and tested on new examples.
6. Explored ablations and hyperparameter trade-offs.
7. Saved and exported a trained model.

### Key Takeaways

- Word embeddings capture semantic similarity better than bag-of-words.
- Masked pooling handles variable-length sequences correctly.
- Early stopping prevents overfitting and saves training time.
- The validation set guides hyperparameter tuning; the test set measures final performance.
- Frozen vs. fine-tuned embeddings is a classic speed/quality trade-off.

### What's Next?

The next notebook tackles the same problem with BERT, a transformer-based pretrained language model. In practice, BERT often beats a small MLP on this kind of task by a few points with less hyperparameter tuning, at the cost of more compute. Understanding the MLP first makes it easier to see what BERT is doing differently.

### Self-Check Quiz

1. Why does `nn.CrossEntropyLoss` accept logits instead of softmax outputs?

   Answer: For numerical stability. Computing `log(softmax(x))` can lose precision due to the `exp()` in softmax. `CrossEntropyLoss` uses `log_softmax`, which is mathematically equivalent but numerically stable via the LogSumExp trick.

2. Your train accuracy is 99% but val accuracy is 88%. What is happening, and how do you fix it?

   Answer: The model is overfitting, memorizing the training set without generalizing. Fixes include increasing dropout from 0.3 to 0.4-0.5, reducing model capacity (smaller `HIDDEN_DIM`), adding more training data, or using L2 weight decay.

3. You freeze Word2Vec embeddings and accuracy drops three points. You unfreeze and it recovers. What does that tell you?

   Answer: The pretrained Word2Vec vectors are useful but not perfectly aligned with this task. Fine-tuning lets the model adapt embeddings (for example, pushing "stock" closer to business-related words in the task-specific space). If freezing had no effect, Word2Vec would already be optimal for this task.

4. Why use a mask during mean pooling?

   Answer: Headlines have variable lengths, and short ones are padded to `MAX_LEN` with `<pad>` tokens (ID 0). Without a mask, padding embeddings would be averaged into the sentence representation and dilute the signal. The mask restricts the average to real tokens.

5. When would you choose mean pooling over max pooling?

   Answer: Mean pooling fits when most words carry useful signal (headlines, reviews). Max pooling fits when a few keywords dominate (sentiment analysis where "excellent" or "terrible" carry most of the signal). For this news task, the category signal is spread across the headline, so mean pooling is a good fit.